# 10 – Data, licensing and ethics section; limitations text; run-metadata annotation

Drafts for the report, **generated from the repository's own files** so that every number is checked against its source. Output: `docs/data_licensing_ethics.md` and `docs/limitations.md` (ready-to-paste text), plus an annotation of the Gemini entries in `dataset/answers/run_metadata.json`.

*Written for: the report's authors (formal first-person plural, to be edited into the final report). The files describe what the repository does; they are not legal advice.*

In [1]:
import json, subprocess, re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

CFG = json.load(open("config.json")); OUT = Path(CFG["analysis_dir"]); DOCS = Path("docs"); DOCS.mkdir(exist_ok=True)
log = json.load(open(CFG["corpus_log_json"])); man = json.load(open(CFG["corpus_manifest_json"]))
scraped = pd.read_csv(CFG["scraped_csv"], keep_default_na=False); urls = pd.read_csv(CFG["urls_full_csv"])
sc = scraped[scraped["url"].isin(urls["url"])]
status = sc["status"].value_counts().to_dict()
sel = dict(log["selection_steps"]); cln = dict(log["cleaning_steps"])
reg = man["articles_per_region"]
pct = lambda a, b: f"{100 * a / b:.0f}%"

## 1. Annotate the Gemini entries in `run_metadata.json` (annotated, not removed)
The file is the record of what was run. The two Gemini entries (a 4-answer smoke test and the full run that hit the free-tier daily quota after 17 more answers) are **annotated as abandoned** and kept; they are not removed. The other entries get a `run_type` label. The step is safe to rerun.

In [2]:
p = Path(CFG["answers_jsonl"]).parent / "run_metadata.json"
meta = json.load(open(p)); smoke_start = {}
for e in meta:                                           # first entry per chatbot = smoke test (LIMIT = 2 prompts), later one = full run
    e.setdefault("run_type", "smoke test (first 2 prompts)" if e["chatbot"] not in smoke_start else "full run")
    smoke_start.setdefault(e["chatbot"], e["run_started_utc"])
for e in meta:
    if e["chatbot"].startswith("gemini"):
        n = 4 if e["run_type"].startswith("smoke") else 17
        e["status"] = "abandoned"
        e["status_reason"] = ("Free-tier daily quota (20 requests per day for this model) was reached after 21 answers in total (4 in the smoke test, 17 in the full run), "
                              "so qwen3.5:9b replaced it as the second chatbot.")
        e["answers_collected_in_this_run"] = n; e["answers_used_in_analysis"] = 0
        e["annotated_on"] = date.today().isoformat()
json.dump(meta, open(p, "w"), indent=2)
answers = pd.read_json(CFG["answers_jsonl"], lines=True)
assert not answers["chatbot"].str.startswith("gemini").any(), "Gemini rows are still in the answers file"
pd.DataFrame([{k: e.get(k) for k in ("chatbot", "run_started_utc", "run_type", "status", "answers_collected_in_this_run")} for e in meta])

,chatbot,run_started_utc,run_type,status,answers_collected_in_this_run
0,gemini-3.8-flash,2026-10-23T12:47:52+00:00,smoke test (first 2 prompts),abandoned,4.0
1,llama3.1:8b,2026-10-23T12:47:52+00:00,smoke test (first 2 prompts),None,NaN
2,gemini-3.8-flash,2026-10-23T12:54:58+00:00,full run,abandoned,17.0
3,llama3.1:8b,2026-10-23T12:54:58+00:00,full run,None,NaN
4,qwen3.5:9b,2026-10-23T15:11:40+00:00,smoke test (first 2 prompts),None,NaN
5,qwen3.5:9b,2026-10-23T15:13:15+00:00,full run,None,NaN


## 2. Data, licensing and ethics section

In [3]:
tracked = subprocess.run(["git", "ls-files"], capture_output=True, text=True).stdout.split()
ignored = [l.strip() for l in open(".gitignore") if l.strip() and not l.startswith("#")]
# no committed CSV may hold article text; list the committed files that carry titles
text_cols, title_cols = [], []
for f in Path("dataset").rglob("*.csv"):
    cols = pd.read_csv(f, nrows=0).columns.str.lower()
    if "text" in cols: text_cols.append(str(f))
    if "title" in cols or "example_title" in cols or "title_a" in cols: title_cols.append(str(f))
assert not text_cols, "a CSV under dataset/ contains article text: " + str(text_cols)
print("git-ignored paths:", ignored); print("files under dataset/ with article titles:", title_cols)

ua = "NCI-student-research/1.0 (contact: <university e-mail address>)"
robots = status.get("robots_blocked", 0); paywall = status.get("paywall", 0)
http_blocked = sum(v for k, v in status.items() if k in ("http_403", "http_451", "http_401", "http_429"))
a_range = log["access_dates"]
lic_llama, lic_qwen = "Llama 3.1 Community License Agreement (Meta)", "Apache License 2.0"

git-ignored paths: ['.env', '.DS_Store', '__pycache__/', '.ipynb_checkpoints/', '.venv/', 'data/']
files under dataset/ with article titles: ['dataset/analysis/near_duplicates_remaining.csv', 'dataset/analysis/dedup_removed_articles.csv', 'dataset/analysis/near_duplicate_clusters.csv', 'dataset/analysis/dataset_label_disagreement_examples.csv']


In [4]:
ethics = f'''# Data, licensing and ethics (draft for the report)

*Drafted from what the repository does, with figures read from its files. Not legal advice; check wording with the module's guidance before submission.*

## 1. News data: source, licence and what we store

**Source.** The URL list comes from the Hugging Face dataset `brewcoua/genai-gdelt-framing` (Couaran, Pathania and Nair, 2026), a derived dataset of {sel["all articles"]:,} online news articles on generative-AI governance (November 2022 to June 2026) built from the GDELT 2.0 Global Knowledge Graph. The dataset is released under **CC BY 4.0** and GDELT under CC BY; we attribute both and cite the paper. The dataset contains URLs and metadata only (`document_id` = URL, month, region, frame flags and embedding scores) and **no article text**. We use the URL, month and region to select articles, and the dataset's own frame labels once, as a cross-check of our dictionary.

**What we did to the data.** We filtered to July 2025 to June 2026 ({sel["months 2025-07 to 2026-06"]:,} articles), kept URLs with an AI term in the path ({sel["AI word in URL path"]:,}), restricted EU and Other to likely-English domains, capped each outlet at 15, and drew a balanced sample of {len(urls):,} URLs (US, UK, EU, Other). After scraping and cleaning the corpus has **{man["rows"]} articles from {man["outlets"]} outlets** (US {reg["US"]}, Other {reg["Other"]}, UK {reg["UK"]}, EU {reg["EU"]}). The corpus is frozen: its SHA-256 hash and counts are recorded in `dataset/news/corpus_manifest.json`.

**Article text belongs to the publishers.** We scraped publicly available pages for non-commercial university research, and we do **not redistribute article text**. The repository therefore stores:

| stored in the repository (committed) | kept only on the authors' machines (git-ignored: `{", ".join(ignored)}`) |
|---|---|
| URL list (URL, month, region, outlet, quarter), sampling and cleaning log, corpus manifest with file hashes | scraped pages and the cleaned corpus (`data/scraped.csv`, `data/news_clean.csv`) |
| per-text frame scores and counts, lexicon, actor and entropy tables, chatbot answers and prompts | the dataset parquet file (about 90 MB, re-downloadable) |
| a few audit tables that carry **article titles and URLs** (never body text): {", ".join(f"`{t}`" for t in title_cols)}; and, as the only body text, about thirty short example quotations (at most about 20 words each) in `07_lexicon_signoff.ipynb`, used to document candidate lexicon terms | validation and stance labelling sheets, review files and low-hit samples, because they contain article text |

The submission ZIP must exclude `data/`. Anyone can rebuild the corpus from the URL list, the notebooks and `config.json`.

## 2. Collection conduct

- **robots.txt.** The scraper reads each site's `robots.txt` (Python `urllib.robotparser`) before every first request to a site and skips URLs it disallows: {robots} of {len(sc):,} sampled URLs were skipped for that reason.
- **Rate limit.** One request per second, one request per URL, no parallel requests. Pages that answered 401, 403, 429 or 451 were not retried or worked around ({http_blocked} URLs). Pages that looked paywalled were dropped ({paywall}); we did not log in or bypass any access control.
- **Identification.** Requests carry the User-Agent `{ua}` so that a site operator can reach us. The address is a university address read from a git-ignored `.env` file and is not published in the repository.
- **Access dates.** {a_range[0][:10]} to {a_range[1][:10]} (UTC), recorded in `dataset/news/corpus_log.json`. An earlier pilot scrape by a team member on 19 and 20 October 2026 (900 URLs, a first version of the corpus that was discarded because only 45 articles were usable) followed the same robots.txt and rate-limit rules and is not part of the analysed data.
- **Outcome of the scrape.** {len(sc):,} URLs requested; {cln["usable text"]:,} gave usable English text ({pct(cln["usable text"], cln["scraped"])}); {cln["on-topic"]} were on-topic; {cln["no duplicates"]} remained after removing duplicates.

## 3. Chatbot answers and terms of use

We generated {len(answers)} answers ({answers["prompt_id"].nunique()} prompts x 2 chatbots x 2 replicates). The prompts are generic policy questions and contain no personal data.
- **Llama 3.1 (8B)** ran locally through Ollama under the {lic_llama}; **Qwen3.5 (9B)** ran locally under the {lic_qwen}. Local models send nothing to a third party. Model digests, versions and settings are recorded in `dataset/answers/run_metadata.json`.
- **Gemini** (`gemini-3.8-flash`, Google AI Studio free tier) was tried first. The free tier allows only 20 requests per day for that model, so the attempt was abandoned after 21 answers, all of which were removed from the analysis; the two attempts stay in `run_metadata.json` annotated as abandoned. Before submission, check the current Gemini API terms: on the free tier Google may use prompts and outputs to improve its products, which is acceptable here only because the prompts are generic policy questions with no personal data.
- Chatbot output is machine-generated text; we analyse it as data and do not present it as the authors' or anyone's opinion.

## 4. Ethical considerations

- **No human participants.** Labelling (frame validation, stance) is done by team members on public texts and on chatbot output. No participant data are collected.
- **Personal data in news text.** Articles name public figures and sometimes private individuals (for example victims in court or safety stories). Article text stays on our machines; committed outputs hold counts, scores, URLs and titles of already published articles. Actor tables list the most mentioned organisations and public figures only.
- **Distressing content.** Some texts discuss suicide, sexual content in the context of chatbot safety, and hate speech. The labelling instructions carry a content note and allow skipping a text.
- **Load on publishers.** About {len(sc):,} page requests were made in two runs (wall-clock span {int((pd.Timestamp(a_range[1]) - pd.Timestamp(a_range[0])).total_seconds() / 3600)} hours including a pause), never faster than one request per second overall, and no site was asked for more than {int(sc.merge(urls[["url", "outlet"]], on="url").groupby("outlet").size().max())} pages.
- **Bias and representativeness.** The corpus is English-language and AI-focused (URL filter), and "Other" means English-language outlets outside the US, EU and UK. Findings describe this corpus, not news in general (see `docs/limitations.md`).
- **Reproducibility.** Seeds, settings, hashes and access dates are recorded; the dataset file is identical in every copy we hold (SHA-256 checked in `04_corpus_audit.ipynb`).
'''
(DOCS / "data_licensing_ethics.md").write_text(ethics); print(len(ethics.split()), "words written to docs/data_licensing_ethics.md")

993 words written to docs/data_licensing_ethics.md


In [5]:
# content note for the labelling instructions (added once)
NOTE = "\n\n**Content note.** Some texts discuss suicide, sexual content in the context of chatbot safety, and hate speech. You may skip any text that distresses you: write `skip` in the label column (the scoring scripts treat it as missing and report how many were skipped) and tell the others.\n"
for f in [Path("dataset/frames/validation_instructions.md"), Path("dataset/frames/stance_codebook.md")]:
    t = f.read_text()
    if "Content note" not in t: f.write_text(t.rstrip() + NOTE)
print("content note present in both instruction files")

content note present in both instruction files


## 3. Limitations text
Five ready-to-paste paragraphs requested by the plan, plus the further limits found by the audits. Every figure is read from the saved result files.

In [6]:
dens = pd.read_csv(OUT / "frame_density.csv").set_index("view")
clf = pd.read_csv(OUT / "style_vs_framing_classifiers.csv")
clusters = pd.read_csv(OUT / "near_duplicate_clusters.csv"); dd = pd.read_csv(OUT / "dedup_before_after_by_region.csv", index_col=0)
quality = pd.read_csv("dataset/frames/review_corpus_quality.csv").set_index("issue")
conf = pd.read_csv(OUT / "dataset_label_comparison_confusion.csv", index_col=0).to_numpy(float)
po = np.trace(conf) / conf.sum(); pe = ((conf.sum(0) / conf.sum()) * (conf.sum(1) / conf.sum())).sum(); ds_acc, ds_kappa = po, (po - pe) / (1 - pe)
ans_len = pd.read_csv(OUT / "length_match_table.csv").set_index("view")["words_mean"]
n_dup_art = int(clusters["size"].sum()); n_extra = n_dup_art - len(clusters)
head_ok = quality.loc[[i for i in quality.index if "first 300 words" in i][0], "chunks_affected"]
auc = lambda feat, split: clf[(clf["features"] == feat) & (clf["split"] == split)]["auc_mean"].iloc[0]
u5 = dens.loc["news (300-word cut)", "share_under_5_hits"]; u5_l = dens.loc["news (length-matched to Llama)", "share_under_5_hits"]; u5_q = dens.loc["news (length-matched to Qwen)", "share_under_5_hits"]
n_under5 = int(round(u5 * man["rows"]))

In [7]:
lim = f'''# Limitations (ready to paste; edit to the report's voice)

*Figures are read from the repository's result files by `10_limitations_and_ethics.ipynb`.*

**1. Two small local models at one temperature.** The chatbot side of the study uses two open-weight models run locally through Ollama: Llama 3.1 8B (4-bit quantised, Q4_K_M) and Qwen3.5 9B (run on Apple's MLX runtime in 4-bit NVFP4, with its reasoning mode switched off so that answers are plain prose). Both were sampled at temperature 0.7 and top-p 0.95, with the replicate number as the random seed and two answers per prompt, giving {len(answers)} answers to 54 prompts that each ask for "about 250 words of plain prose, without headings or bullet points". Our results therefore describe these two models under these settings, not "AI chatbots" in general: larger hosted models, other temperatures, system prompts, or users who ask follow-up questions may produce different frame mixes. Quantisation can also change outputs slightly. The two models differ in length (mean {ans_len["Llama"]:.0f} words for Llama and {ans_len["Qwen"]:.0f} for Qwen against the requested 250), which is why every comparison is made on texts cut to 300 words and at a matched number of frame hits.

**2. Gemini was dropped.** We first collected answers from Google's `gemini-3.8-flash` through the free tier of Google AI Studio, but the free quota (20 requests per day for that model) was exhausted after 21 answers (4 in a smoke test and 17 in the full run). We replaced it with Qwen3.5 9B, which also let us keep the design of two chatbots and two replicates. The 21 partial Gemini answers were removed from the analysis file. `dataset/answers/run_metadata.json` still lists the two Gemini attempts; they are **annotated as abandoned** (not deleted), so the record of what was run is complete. The consequence is that no large hosted commercial model is included; the plan already named the second chatbot as the first item to cut under time pressure.

**3. The AI-word URL filter skews the corpus towards AI-focused coverage.** To get enough on-topic articles from a 1.1-million-row URL list that has no article text, we kept only URLs with an AI term in the path (for example `ai`, `chatgpt`, `openai`, `chatbot`): {sel["AI word in URL path"]:,} of {sel["regions US, UK, EU, Other"]:,} URLs in the July 2025 to June 2026 window. An earlier attempt without this filter (900 URLs, April to June 2026, three regions) gave only 45 usable articles; with the filter the pilot on-topic rate was {log["pilot"]["on_topic"]:.0%}. The price is selection bias: articles in which AI is a side topic, and outlets that do not put AI in their URLs, are missing, so the frame distributions describe AI-focused coverage, not all news that mentions AI regulation. The filter is applied identically to every region, so regional comparisons are not distorted by it, but absolute frame emphasis may be. EU and Other were also restricted to likely-English domains, each outlet was capped at 15 articles, and the window is twelve months, not the dataset's full 2022 to 2026 span.

**4. Sample sizes are modest and uneven.** The frozen corpus has {man["rows"]} articles from {man["outlets"]} outlets (US {reg["US"]}, Other {reg["Other"]}, UK {reg["UK"]}, EU {reg["EU"]}); the largest outlet contributes {man["max_articles_per_outlet"]} articles. The EU and UK strata are the smallest ({reg["EU"]} and {reg["UK"]}), so every EU and UK interval is wide and the region comparison (H3, exploratory by design) is weakest exactly where policy interest is highest. Duplicates matter too: the exact-duplicate rule removed {int(dd.loc["All", "removed"])} of {int(dd.loc["All", "before"])} on-topic articles ({dd.loc["All", "removed_%"]:.0f}%; 83 of the 84 were copies of the same story on a different outlet, one on a sister site of the same publisher), but {len(clusters)} clusters of near-duplicate articles ({n_dup_art} articles; {n_extra} redundant) remain in the frozen corpus, concentrated in a few news events, so articles are not fully independent. "Other" means English-language outlets outside the US, EU and UK and says nothing about the rest of the world. Intervals therefore use a cluster bootstrap (outlet for news, question for chatbots), and chatbot intervals rest on only 18 question clusters.

**5. A quarter of news chunks have too few frame hits to score, and how we handle it.** Frame entropy needs enough hits to be meaningful. At the 300-word cut {n_under5} of {man["rows"]} news chunks ({u5:.0%}) have fewer than 5 hits (none of the chatbot answers do), and the share rises to {u5_l:.0%} when news is cut to Llama's shorter answer lengths and {u5_q:.0%} for Qwen's. A plain entropy comparison would be biased twice: texts with few hits have a lower entropy by arithmetic, and dropping them would keep only the most frame-dense articles. We therefore (i) report the share of texts under 5 hits as a result in its own right (frame density), (ii) compare groups on rarefied entropy, drawing 5 hits at random without replacement 1,000 times per text, using only texts with at least 5 hits, (iii) bound the effect of the excluded texts by scoring them at the minimum, at the group median and at the maximum possible entropy, (iv) repeat at 3 and 8 hits, and (v) run a pooled version that sums the hits of all texts in a group so that every text is used. A related limit is that the on-topic rule was applied to the whole article while frames are scored on its first 300 words: only {head_ok} of {man["rows"]} corpus articles ({head_ok / man["rows"]:.0%}) still satisfy the on-topic rule within those 300 words, so much of the regulatory content of an article sits beyond the part that is scored.

## Further limits found while checking the work

- **The dictionary measures words, not meaning.** It cannot tell "regulation protects people" from "regulation burdens firms"; frames are emphasis, not stance. The lexicon was refined once, after an audit of hit counts on the chatbot answers and before any hypothesis test; the changes are in `dataset/frames/CHANGELOG.md`. Topic-defining words (regulate, law, rule, government, policy) are left out on purpose because they occur in almost every text.
- **Our dictionary and the dataset's own frame labels disagree.** On the {int(conf.sum())} corpus articles where the dataset has a dominant frame, our top frame agrees with it in {ds_acc:.0%} of cases (Cohen's kappa {ds_kappa:.2f}). The two instruments read different material (the dataset uses GDELT entity names and quotations, we use article text), so neither is the truth; human validation of 50 chunks is the real test and is still to be done by the team.
- **Style differs far more than framing.** A classifier using only function words, punctuation and formatting separates chatbot answers from news with an AUC of {auc("A style (all texts)", "outlets/questions held out"):.2f} on held-out outlets and questions; a classifier using only the six frame proportions reaches {auc("B frame mix (texts with >= 5 hits)", "outlets/questions held out"):.2f}. Formatting features (paragraph breaks, digits) do not overlap between the two sources, so a matched subset cannot remove them; entropy differences must be read with this in mind.
- **Text quality.** Scraped text contains some boilerplate, leftover markup and encoding errors ({quality.iloc[0]["share"]:.0%} of chunks carry subscription or sign-up text), which dilutes frame density slightly.
- **Stance labelling and actors.** Stance labels rest on two labellers and 20 shared texts, so kappa has a wide interval. Named-entity recognition uses a small general model and an alias table that covers the most frequent actors only; chatbot answers name almost no actors.
'''
(DOCS / "limitations.md").write_text(lim); print(len(lim.split()), "words written to docs/limitations.md")

1198 words written to docs/limitations.md


In [8]:
# every figure quoted in the limitations text, in one place for checking
pd.Series({"answers": len(answers), "news chunks": man["rows"], "outlets": man["outlets"], "news under 5 hits (300-word cut)": f"{n_under5} ({u5:.1%})",
           "under 5 hits, matched to Llama / Qwen": f"{u5_l:.1%} / {u5_q:.1%}", "near-duplicate clusters / articles / redundant": f"{len(clusters)} / {n_dup_art} / {n_extra}",
           "articles still on-topic within 300 words": f"{head_ok} ({head_ok / man['rows']:.1%})", "dataset-label agreement / kappa": f"{ds_acc:.3f} / {ds_kappa:.3f}",
           "Llama / Qwen mean words": f"{ans_len['Llama']:.0f} / {ans_len['Qwen']:.0f}"})

answers                                                     216
news chunks                                                 420
outlets                                                     285
news under 5 hits (300-word cut)                    108 (25.7%)
under 5 hits, matched to Llama / Qwen             38.6% / 27.1%
near-duplicate clusters / articles / redundant     14 / 33 / 19
articles still on-topic within 300 words            157 (37.4%)
dataset-label agreement / kappa                   0.352 / 0.197
Llama / Qwen mean words                               223 / 287
dtype: object